# 00 · Parse SEC 10-K HTML into structured elements (pre-chunking)

**Goal:** turn raw 10-K HTML into a clean, *source-agnostic* list of elements (`heading`, `paragraph`, `table`) with metadata, so the later chunking/embedding stages never need to know which company or HTML generator produced the file.

```
raw_html ─► metadata + XBRL facts ─► strip noise ─► blocks + sections
         ─► per table: grid ─► cell typing/merge ─► header detect ─► column groups
                       ─► hierarchy ─► title/units ─► XBRL validation ─► quality score
         ─► data/parsed/<doc_id>.elements.jsonl  (+ facts.jsonl, quality_report.csv)
```

**Guiding rule:** never write rules per company or per column position. Every rule uses signals that hold for any filer: what a cell *contains* (number, `$`, `(`, year, text), how it *looks* (bold, indent), and what XBRL says it *is*.

## 1. Setup
**What:** imports, project paths, output folder.

**Why:**
- `BeautifulSoup` with the **`lxml`** parser: fast, and tolerant of the messy XHTML that Workiva produces. Namespaced Inline-XBRL tags like `ix:nonFraction` come through as plain lowercase tag names (`ix:nonfraction`), so they're easy to query. A strict XML parser would force namespace handling and break on HTML quirks. bs4 warns "XML parsed as HTML" because the file starts with `<?xml?>`; that choice is deliberate, so the warning is silenced.
- `pandas` is used only for *inspection* in this notebook. The pipeline itself works on plain dataclasses, so it can later move into `src/` without depending on pandas.
- Paths are resolved relative to the project root, so the notebook works whether Jupyter starts in `exploration/` or in the root.

In [7]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path

import warnings

import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

ROOT = Path.cwd().parent if Path.cwd().name == "exploration" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw_html"
OUT_DIR = ROOT / "data" / "parsed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = sorted(RAW_DIR.glob("*.html"))
[f.name for f in FILES]

['Apple_Inc__10K_2025_10_31.html',
 'Microsoft_Corp__10K_2026_06_30.html',
 'Tesla_Inc__10K_2026_01_29.html']

## 2. Configuration: every tunable value in one place
**What:** thresholds and regex patterns.

**Why:** when a new company breaks something, you should be able to fix it by adjusting **config** (and adding a regression test), not by adding `if company == "X"` branches. Each value below is a *generic* signal:
- `boilerplate_*`: a short line repeated ≥5 times (with digits masked) is a page header or footer, e.g. `Apple Inc. | 2025 Form 10-K | 21`.
- `nbsp_pt`: one `&nbsp;` counts as about 2.5pt of indentation. Apple indents with `&nbsp;`, Tesla with `padding-left`; both are converted to points so they can be compared.
- `UNITS_RE`: units appear as "(In millions…)", "(dollars in millions)", "(in thousands)" and so on.
- `ITEM_RE` / `PART_RE`: 10-K sections are legally standardized ("Item 1A", "Item 7", …), so section detection works the same for every filer.

In [8]:
CONFIG = {
    "boilerplate_min_repeats": 5,
    "boilerplate_max_len": 80,
    "heading_max_len": 150,
    "title_lookback": 4,
    "max_header_rows": 6,
    "nbsp_pt": 2.5,
    "indent_tolerance_pt": 1.0,
    "numeric_table_ratio": 0.3,
    "note_min_len": 100,
    "low_confidence": 0.6,
}
DASHES = {"—", "–", "-", "−", "--"}
VALUE_TYPES = {"NUMBER", "DASH", "YEAR"}
BOLD_WEIGHTS = {"700", "800", "900", "bold", "bolder"}
BLOCK_TAGS = ["div", "p", "table", "h1", "h2", "h3", "h4", "h5", "h6", "li", "center"]
SCALE_TO_UNITS = {3: "in thousands", 6: "in millions", 9: "in billions"}

WS_RE = re.compile(r"\s+")
UNITS_RE = re.compile(r"\b(?:in|dollars in|amounts in|\$ in)\s+(millions|thousands|billions)\b", re.I)
ITEM_RE = re.compile(r"^item\s+(\d{1,2}[a-c]?)\s*[.:\-–—]?\s*(.*)$", re.I)
PART_RE = re.compile(r"^part\s+(iv|i{1,3})\b\.?\s*(.*)$", re.I)
YEAR_RE = re.compile(r"^(?:fiscal\s+|fy\s*)?(?:19|20)\d{2}$", re.I)
DATE_RE = re.compile(
    r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2}\b|\b(year|years|quarter|quarters|months|weeks)\s+ended\b",
    re.I,
)
CURRENCY_RE = re.compile(r"^(?:US\$|C\$|A\$|\$|€|£|¥)$")
NUM_CORE_RE = re.compile(r"^\d[\d,]*(?:\.\d+)?$|^\.\d+$")
UNITS_ONLY_RE = re.compile(r"^\(?\s*(?:\$|dollars|amounts)?\s*in\s+(?:millions|thousands|billions)\b[^)]*\)?$", re.I)
MAGNITUDES = {"thousand": 1e3, "million": 1e6, "billion": 1e9, "trillion": 1e12}
MAGNITUDE_RE = re.compile(r"(thousand|million|billion|trillion)s?$", re.I)
PAGE_NUM_RE = re.compile(r"^(?:page\s*)?(?:\d{1,3}|[ivxlc]{1,6})$|^[-–—]\s*\d{1,3}\s*[-–—]$", re.I)

## 3. Data model: the contract between pipeline stages
**What:** small dataclasses for `Document → Element → Table → Row → Cell`, plus `XbrlFact`.

**Why:**
- This is the **most important production decision** in the notebook. Every later stage (chunking, embedding, evaluation) reads *only* this model. Supporting a new source (PDF, another filing vendor) then means writing one new adapter that fills the model; nothing downstream changes.
- `Row` holds `label`, `level` and `path` (e.g. `["Net sales", "Products"]`), so the parent/child structure that Markdown loses is kept explicitly.
- `Cell` holds both the display `text` and the parsed `value`, plus `fact_ids` linking back to XBRL. You can show the LLM exactly what the filing shows, and still compute with exact numbers.
- `Table.quality` makes parsing problems visible instead of silent.

In [9]:
@dataclass
class XbrlFact:
    id: str
    concept: str
    context: str
    unit: str | None
    scale: int
    sign: int
    shown: float | None  # absolute number as printed in the document
    value: float | None  # true value = sign * shown * 10**scale


@dataclass
class Cell:
    text: str
    ctype: str  # NUMBER | DASH | YEAR | DATE | TEXT | EMPTY ...
    value: float | None = None
    is_percent: bool = False
    currency: str | None = None
    fact_ids: list[str] = field(default_factory=list)


@dataclass
class Column:
    name: str
    period_start: str | None = None
    period_end: str | None = None


@dataclass
class Row:
    label: str
    level: int
    path: list[str]
    row_type: str  # data | heading | note
    is_total: bool
    cells: list[Cell]  # aligned with Table.columns[1:]


@dataclass
class Table:
    table_id: str
    kind: str  # financial | textual | text_block | layout | toc
    title: str | None = None
    units: str | None = None
    intro: str | None = None
    columns: list[Column] = field(default_factory=list)
    rows: list[Row] = field(default_factory=list)
    period_axis: str | None = None
    quality: dict = field(default_factory=dict)
    markdown: str = ""


@dataclass
class Element:
    doc_id: str
    order: int
    type: str  # heading | paragraph | table
    part: str | None
    section: str | None
    section_title: str | None
    text: str | None = None
    table: Table | None = None


@dataclass
class Document:
    doc_id: str
    company: str
    ticker: str | None
    cik: str | None
    form: str
    fiscal_year: int | None
    period_end: str | None
    source_file: str
    elements: list[Element] = field(default_factory=list)

## 4. Low-level helpers: text, CSS, bold, indent
**What:** whitespace normalization, CSS `style` parsing, point conversion, bold detection.

**Why:**
- `clean()` turns `&nbsp;` and pretty-printing newlines into single spaces. The HTML source wraps text like `Apple\n            Inc.`, which would otherwise end up in your embeddings.
- `padding_left_pt()` understands CSS shorthand (`padding: 2px 1pt 2px 10pt` means left = 10pt). Tesla encodes row indentation this way, so this is required to recover hierarchy.
- `is_bold()` checks **every text node** and its ancestors. Workiva puts styling on nested `<span>`s, not on the `<td>`, so only checking the outer tag would miss it. A block counts as bold only if *all* of its text is bold. That stops a paragraph with one bold word from being treated as a heading.

In [10]:
def clean(s: str | None) -> str:
    return WS_RE.sub(" ", (s or "").replace("\xa0", " ")).strip()


def style_of(tag) -> dict:
    out = {}
    for part in (tag.get("style") or "").split(";"):
        if ":" in part:
            k, v = part.split(":", 1)
            out[k.strip().lower()] = v.strip().lower()
    return out


def to_pt(v: str) -> float:
    m = re.match(r"(-?[\d.]+)\s*(pt|px|em|in)?", v or "")
    if not m:
        return 0.0
    n, unit = float(m.group(1)), m.group(2) or "px"
    return {"px": n * 0.75, "em": n * 10, "in": n * 72}.get(unit, n)


def padding_left_pt(st: dict) -> float:
    if "padding-left" in st:
        return to_pt(st["padding-left"])
    parts = st.get("padding", "").split()
    idx = {1: 0, 2: 1, 3: 1, 4: 3}.get(len(parts))
    return to_pt(parts[idx]) if idx is not None else 0.0


def is_bold(tag) -> bool:
    nodes = [s for s in tag.find_all(string=True) if clean(s)]
    if not nodes:
        return False
    for s in nodes:
        p, bold = s.parent, False
        while p is not None:
            if p.name in ("b", "strong") or style_of(p).get("font-weight") in BOLD_WEIGHTS:
                bold = True
                break
            if p is tag:
                break
            p = p.parent
        if not bold:
            return False
    return True

## 5. Load a filing and read document metadata from XBRL `dei:` tags
**What:** parse the file, then read company, ticker, CIK, form type and fiscal year.

**Why:**
- Metadata comes from the **`dei:` XBRL cover-page tags** (`dei:EntityRegistrantName`, `dei:DocumentFiscalYearFocus`, …), *not* from the filename. Every SEC filer must tag these, so this works for any company. The filename is only a fallback.
- Apple's fiscal year ends in September and Tesla's in December. Reading the fiscal year from XBRL avoids guessing it from dates.
- These fields become metadata on every chunk later (filtering by company/year is often more useful than vector similarity).

In [11]:
def load(path: Path) -> BeautifulSoup:
    return BeautifulSoup(path.read_bytes(), "lxml")


def doc_metadata(soup: BeautifulSoup, path: Path) -> dict:
    dei = {}
    for t in soup.find_all("ix:nonnumeric"):
        name = t.get("name", "")
        if name.startswith("dei:") and name not in dei:
            dei[name] = clean(t.get_text())
    company = dei.get("dei:EntityRegistrantName") or path.stem.split("__")[0].replace("_", " ")
    ticker = dei.get("dei:TradingSymbol")
    form = dei.get("dei:DocumentType", "10-K")
    fy = dei.get("dei:DocumentFiscalYearFocus", "")
    return {
        "doc_id": f"{ticker or re.sub(r'\W+', '_', company)}_{form}_{fy}",
        "company": company,
        "ticker": ticker,
        "cik": dei.get("dei:EntityCentralIndexKey"),
        "form": form,
        "fiscal_year": int(fy) if fy.isdigit() else None,
        "period_end": dei.get("dei:DocumentPeriodEndDate"),
        "source_file": path.name,
    }


soup_apple = load(FILES[0])
pd.DataFrame([doc_metadata(load(f), f) for f in FILES])

,doc_id,company,ticker,cik,form,fiscal_year,period_end,source_file
0,AAPL_10-K_2025,Apple Inc.,AAPL,0000320193,10-K,2025,"September 27, 2025",Apple_Inc__10K_2025_10_31.html
1,MSFT_10-K_2026,MICROSOFT CORPORATION,MSFT,0000789019,10-K,2026,"June 30, 2026",Microsoft_Corp__10K_2026_06_30.html
2,TSLA_10-K_2025,"Tesla, Inc.",TSLA,0001318605,10-K,2025,"December 31, 2025",Tesla_Inc__10K_2026_01_29.html


## 6. Extract XBRL contexts and numeric facts
**What:** map each `xbrli:context` to its period and dimensions, and turn each `ix:nonFraction` into an `XbrlFact`.

**Why:**
- Every reported number in the filing is tagged with *what it is* (`concept`), *when* (`contextRef` → period), *scale* (`scale="6"` = millions) and *sign* (`sign="-"`). This is the **ground truth** for numbers.
- We use it two ways: (1) as a separate facts table for exact numeric Q&A later, and (2) to **validate** our own table parsing (step 13).
- This has to happen **before** removing hidden content, because contexts live inside the hidden `ix:header` block.
- `shown` (as printed) and `value` (the true value) are kept separately: `307,003` with scale 6 is 307,003,000,000.

In [12]:
def parse_contexts(soup: BeautifulSoup) -> dict:
    out = {}
    for c in soup.find_all("xbrli:context"):
        def get(name):
            t = c.find(name)
            return t.get_text(strip=True) if t else None

        inst = get("xbrli:instant")
        out[c.get("id")] = {
            "start": get("xbrli:startdate"),
            "end": inst or get("xbrli:enddate"),
            "instant": inst is not None,
            "dims": {m.get("dimension"): clean(m.get_text()) for m in c.find_all(["xbrldi:explicitmember", "xbrldi:typedmember"])},
        }
    return out


def parse_fact(t) -> XbrlFact:
    txt, fmt = clean(t.get_text()), t.get("format", "")
    scale, sign = int(t.get("scale") or 0), (-1 if t.get("sign") == "-" else 1)
    if "zero" in fmt or txt in DASHES:
        shown = 0.0
    else:
        core = re.sub(r"[^\d.,]", "", txt)
        core = core.replace(".", "").replace(",", ".") if "comma-decimal" in fmt else core.replace(",", "")
        try:
            shown = float(core)
        except ValueError:
            shown = None
    value = None if shown is None else sign * shown * 10**scale
    return XbrlFact(t.get("id"), t.get("name"), t.get("contextref"), t.get("unitref"), scale, sign, shown, value)


def facts_frame(facts: dict, contexts: dict) -> pd.DataFrame:
    rows = []
    for f in facts.values():
        ctx = contexts.get(f.context, {})
        rows.append({**asdict(f), "period_start": ctx.get("start"), "period_end": ctx.get("end"),
                     "instant": ctx.get("instant"), "dims": ctx.get("dims") or {}})
    return pd.DataFrame(rows)


contexts_apple = parse_contexts(soup_apple)
facts_apple = {f.id: f for f in map(parse_fact, soup_apple.find_all("ix:nonfraction"))}
print(f"contexts={len(contexts_apple)}  numeric facts={len(facts_apple)}")
fa = facts_frame(facts_apple, contexts_apple)
fa[fa.concept.eq("us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax") & fa.dims.map(len).eq(0)][
    ["concept", "shown", "scale", "value", "period_start", "period_end"]]

contexts=182  numeric facts=969


,concept,shown,scale,value,period_start,period_end
8,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,416161.0,6,4.161610e+11,2024-09-29,2025-09-27
9,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,391035.0,6,3.910350e+11,2023-10-01,2024-09-28
10,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,383285.0,6,3.832850e+11,2022-09-25,2023-09-30
306,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,416161.0,6,4.161610e+11,2024-09-29,2025-09-27
307,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,391035.0,6,3.910350e+11,2023-10-01,2024-09-28
308,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,383285.0,6,3.832850e+11,2022-09-25,2023-09-30
867,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,416161.0,6,4.161610e+11,2024-09-29,2025-09-27
896,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,391035.0,6,3.910350e+11,2023-10-01,2024-09-28
925,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,383285.0,6,3.832850e+11,2022-09-25,2023-09-30
958,us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax,416161.0,6,4.161610e+11,2024-09-29,2025-09-27


## 7. Remove noise: hidden XBRL header, page headers/footers, `<br>`
**What:** delete `display:none` blocks and absolutely positioned page header/footer containers, and turn `<br>` into spaces. Then (in step 8) drop short lines that repeat many times.

**Why:**
- The hidden `ix:header` contains thousands of context definitions. Without removing it, they'd show up as "text" and get embedded.
- Page footers such as `Apple Inc. | 2025 Form 10-K | 21` appear on every page. Embedded, they'd create dozens of near-identical, useless chunks, and they interrupt sentences that span a page break.
- Two layers of protection: **(a)** structural (Workiva puts footers in `position:absolute` containers), and **(b)** generic, vendor-independent *repetition* detection (step 8). Layer (b) also catches footers from other HTML generators.
- `<br>` → space keeps `September 27,<br/>2025` from collapsing into `September 27,2025`.

In [13]:
def strip_noise(soup: BeautifulSoup) -> Counter:
    removed = Counter()
    for t in soup.find_all(style=re.compile(r"display\s*:\s*none", re.I)):
        removed["hidden_blocks"] += 1
        t.decompose()
    for t in soup.find_all(style=re.compile(r"position\s*:\s*absolute", re.I)):
        if t.parent is not None:
            removed["page_header_footer_blocks"] += 1
            t.decompose()
    for br in soup.find_all("br"):
        br.replace_with(" ")
    for block in soup.find_all(["div", "p", "li"]):  # stacked divs inside one cell must not glue words together
        block.append(" ")
    return removed


def norm_key(t: str) -> str:
    return re.sub(r"\d+", "#", t.lower())


def find_boilerplate(texts: list[str]) -> set[str]:
    counts = Counter(norm_key(t) for t in texts if len(t) <= CONFIG["boilerplate_max_len"])
    return {k for k, n in counts.items() if n >= CONFIG["boilerplate_min_repeats"]}


def is_boilerplate(text: str, boilerplate: set[str], bold: bool = False) -> bool:
    if PAGE_NUM_RE.match(text):
        return True
    if UNITS_RE.search(text):  # "(In millions)" repeats a lot but carries meaning
        return False
    if bold and (ITEM_RE.fullmatch(text) or PART_RE.fullmatch(text)):
        return False
    return len(text) <= CONFIG["boilerplate_max_len"] and norm_key(text) in boilerplate


strip_noise(soup_apple)

Counter({'page_header_footer_blocks': 60, 'hidden_blocks': 33})

## 8. Walk the document in reading order → blocks, then detect 10-K sections
**What:** produce a flat, ordered list of *leaf* text blocks and *top-level* tables. Track `Part` / `Item` from bold headings.

**Why:**
- **Leaf blocks only** (a `div` with no nested block): Workiva nests `div`s, and taking outer divs would duplicate text.
- **Top-level tables only**: text inside a table is handled by the table parser, never duplicated as a paragraph.
- **Sections come from bold, short lines matching `Item N`**. Requiring bold and short skips both the *table of contents* (a table, handled separately) and cross-references inside sentences ("…see Part II, Item 7 of this Form 10-K…").
- The section is stored on every element, so you can later filter retrieval by section (e.g. Item 1A *Risk Factors* only).

In [14]:
def iter_blocks(body):
    for el in body.find_all(BLOCK_TAGS):
        in_table = el.find_parent("table") is not None
        if el.name == "table":
            if not in_table:
                yield el
        elif not in_table and el.find(BLOCK_TAGS) is None and clean(el.get_text("")):
            yield el


def detect_section(text: str, bold: bool):
    if not bold or len(text) > CONFIG["heading_max_len"]:
        return None
    if m := ITEM_RE.match(text):
        return "item", f"Item {m.group(1).upper()}", clean(m.group(2)) or None
    if m := PART_RE.match(text):
        return "part", f"Part {m.group(1).upper()}", clean(m.group(2)) or None
    return None


blocks_apple = list(iter_blocks(soup_apple.body))
texts = [clean(b.get_text("")) for b in blocks_apple if b.name != "table"]
bp_apple = find_boilerplate(texts)
print(f"blocks={len(blocks_apple)}  tables={sum(b.name == 'table' for b in blocks_apple)}  boilerplate patterns={sorted(bp_apple)}")
pd.DataFrame([(t, s) for b, t in zip([b for b in blocks_apple if b.name != "table"], texts)
              if (s := detect_section(t, is_bold(b)))], columns=["text", "detected"]).head(25)

blocks=648  tables=62  boilerplate patterns=['apple inc.', 'see accompanying notes to consolidated financial statements.']


,text,detected
0,PART I,"(part, Part I, None)"
1,Item 1. Business,"(item, Item 1, Business)"
2,Item 1A. Risk Factors,"(item, Item 1A, Risk Factors)"
3,Item 1B. Unresolved Staff Comments,"(item, Item 1B, Unresolved Staff Comments)"
4,Item 1C. Cybersecurity,"(item, Item 1C, Cybersecurity)"
5,Item 2. Properties,"(item, Item 2, Properties)"
6,Item 3. Legal Proceedings,"(item, Item 3, Legal Proceedings)"
7,Item 4. Mine Safety Disclosures,"(item, Item 4, Mine Safety Disclosures)"
8,PART II,"(part, Part II, None)"
9,"Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases ...","(item, Item 5, Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Pur..."


## 9. Table step 1: build a real grid (expand `colspan`/`rowspan`)
**What:** for every `<td>`, record its text, **start column**, **span**, bold, indent (in points), XBRL fact ids, and whether it contains an internal `#` link.

**Why:**
- Physical widths vary from 3 to 57 columns, so position-based rules can't work. Each cell's true `[start, start+span)` interval in a rectangular grid is the coordinate system the later steps rely on.
- `rowspan` cells are **filled down** into the rows they span (both files use `rowspan`). Apple's lease table has "Operating leases" spanning 2 rows; without fill-down the second row loses its label. The copies carry no XBRL ids, so facts are never double-counted.
- Indent = `padding-left` (Tesla style) **+** leading `&nbsp;` count (Apple style) **+** inner `div` padding. Both encodings map to one number.
- Fact ids are captured *per cell*, so each printed number keeps its exact XBRL meaning.

In [15]:
@dataclass
class RawCell:
    text: str
    start: int
    span: int
    bold: bool
    indent_pt: float
    fact_ids: list[str]
    has_link: bool
    ctype: str = ""


def raw_rows(table) -> list[list[RawCell]]:
    rows, carried = [], defaultdict(dict)  # row -> {col: copy of a rowspan cell from above}
    trs = [tr for tr in table.find_all("tr") if tr.find_parent("table") is table]
    for r, tr in enumerate(trs):
        col, cells = 0, []
        for td in tr.find_all(["td", "th"], recursive=False):
            while col in carried[r]:
                cells.append(carried[r].pop(col))
                col = cells[-1].start + cells[-1].span
            cs, rs = max(1, int(td.get("colspan") or 1)), max(1, int(td.get("rowspan") or 1))
            raw = td.get_text("")
            lead = raw.strip(" \t\r\n")
            nbsp = len(lead) - len(lead.lstrip("\xa0"))
            text = clean(raw)
            inner_pad = max((padding_left_pt(style_of(d)) for d in td.find_all(["div", "p"])), default=0.0)
            cells.append(RawCell(
                text=text, start=col, span=cs, bold=bool(text) and is_bold(td),
                indent_pt=padding_left_pt(style_of(td)) + inner_pad + nbsp * CONFIG["nbsp_pt"],
                fact_ids=[f.get("id") for f in td.find_all("ix:nonfraction")],
                has_link=td.find("a", href=re.compile(r"^#")) is not None,
            ))
            for rr in range(r + 1, r + rs):  # fill down: the spanned rows show this text too (no fact ids: no double count)
                c = cells[-1]
                carried[rr][col] = RawCell(c.text, c.start, c.span, c.bold, c.indent_pt, [], c.has_link)
            col += cs
        cells += carried.pop(r, {}).values()
        rows.append(sorted(cells, key=lambda c: c.start))
    return rows


def find_table_after(blocks, heading_pattern: str):
    """Demo helper: first top-level table after a non-table block matching the pattern."""
    seen = False
    for b in blocks:
        if b.name != "table" and re.search(heading_pattern, clean(b.get_text("")), re.I):
            seen = True
        elif b.name == "table" and seen:
            return b


demo_table = find_table_after(blocks_apple, r"^CONSOLIDATED STATEMENTS OF OPERATIONS$")
demo_raw = raw_rows(demo_table)
print("physical width:", max(sum(c.span for c in r) for r in demo_raw), "| rows:", len(demo_raw))
for r in demo_raw[1:5]:
    print([(c.start, c.span, c.text) for c in r if c.text])

physical width: 18 | rows: 32
[(3, 15, 'Years ended')]
[(3, 3, 'September 27, 2025'), (9, 3, 'September 28, 2024'), (15, 3, 'September 30, 2023')]
[(0, 3, 'Net sales:')]
[(0, 3, 'Products'), (3, 1, '$'), (4, 1, '307,003'), (9, 1, '$'), (10, 1, '294,866'), (15, 1, '$'), (16, 1, '298,085')]


## 10. Table step 2: type each cell and merge fragments
**What:** classify every non-empty cell (`CURRENCY`, `OPEN`, `SUFFIX`, `NUMBER`, `DASH`, `YEAR`, `DATE`, `TEXT`), then glue fragments back together: `$` + `307,003` → `$307,003`, and `(` + `1,234` + `)` → `(1,234)` = **−1234**.

**Why:**
- Filers print `$`, `(`, `)` and `%` in **separate cells** for alignment. Without merging, negatives lose their sign and every value column turns into three columns.
- The rules use **only the cell's content**, never its position, so they work for any layout.
- `YEAR` is a separate type from `NUMBER` because `2025` in a header means a period, not a value. Header detection (next step) depends on that difference.
- A dash `—` means zero or nil in financial tables. It gets `value=0` but keeps type `DASH`, so the original meaning is preserved.

In [16]:
def parse_number(t: str):
    """-> (value, is_percent, currency, is_dash)"""
    s = t.replace("−", "-").replace(" ", "")
    m = re.search(r"US\$|C\$|A\$|\$|€|£|¥", s)
    cur = m.group(0) if m else None
    if cur:
        s = s.replace(cur, "")
    pct = "%" in s
    s = s.replace("%", "")
    neg = s.startswith("(")
    s = s.strip("()")
    if s in DASHES:
        return 0.0, pct, cur, True
    if s.startswith("-"):
        neg, s = True, s[1:]
    mult = 1.0
    if m := MAGNITUDE_RE.search(s):  # "$2.0 trillion"
        mult, s = MAGNITUDES[m.group(1).lower()], s[: m.start()]
    if NUM_CORE_RE.match(s):
        v = float(s.replace(",", "")) * mult
        return (-v if neg else v), pct, cur, False
    return None, pct, cur, False


def cell_type(t: str) -> str:
    if not t:
        return "EMPTY"
    if CURRENCY_RE.match(t):
        return "CURRENCY"
    if t in ("(", "$(", "($"):
        return "OPEN"
    if t in (")", "%", ")%", "%)"):
        return "SUFFIX"
    if YEAR_RE.match(t):
        return "YEAR"
    v, _, _, dash = parse_number(t)
    if dash:
        return "DASH"
    if v is not None:
        return "NUMBER"
    if len(t) <= 60 and DATE_RE.search(t):
        return "DATE"
    return "TEXT"


def merge_fragments(cells: list[RawCell]) -> list[RawCell]:
    out: list[RawCell] = []
    for c in (c for c in cells if c.text):
        c.ctype = cell_type(c.text)
        p = out[-1] if out else None
        glue = p is not None and (
            (p.ctype in ("CURRENCY", "OPEN") and c.ctype in VALUE_TYPES | {"CURRENCY", "OPEN"})
            or (c.ctype == "SUFFIX" and p.ctype in VALUE_TYPES)
        )
        if glue:
            end = max(p.start + p.span, c.start + c.span)
            p.text, p.span = p.text + c.text, end - p.start
            p.fact_ids += c.fact_ids
            p.ctype = cell_type(p.text)
            continue
        out.append(c)
    return out


demo_merged = [merge_fragments(r) for r in raw_rows(demo_table)]
products = next(r for r in demo_merged if r and r[0].text == "Products")
print("BEFORE:", [c.text for c in next(r for r in demo_raw if r and r[0].text == "Products") if c.text])
print("AFTER :", [(c.text, c.ctype, c.start, c.span) for c in products])

BEFORE: ['Products', '$', '307,003', '$', '294,866', '$', '298,085']
AFTER : [('Products', 'TEXT', 0, 3), ('$307,003', 'NUMBER', 3, 2), ('$294,866', 'NUMBER', 9, 2), ('$298,085', 'NUMBER', 15, 2)]


## 11. Table step 3: classify the table, then detect header rows
**What:**
- **Classify** each table as `layout` (empty), `toc` (mostly `#` links, i.e. the table of contents), `financial` (has XBRL facts or ≥30% numeric cells) or `textual`.
- **Header rows** = top rows that contain no XBRL facts and no plain numbers, **and** either have an empty label column, are all bold, or contain years/dates.

**Why:**
- Different table types need different treatment. Embedding the TOC adds noise ("Item 7 … 21"), and layout tables carry nothing.
- There are **no `<th>` tags** in either file, so headers must be inferred. "First row with numbers = data" fails because `2025` looks like a number, which is why `YEAR`/`DATE` are separate types.
- **An XBRL fact in a row = definitely data.** It's the strongest available signal and works for every filer that uses Inline XBRL (all US filers since 2019–2021).
- "Empty label column" is the most common visual header pattern across filers, e.g. `| | 2025 | 2024 |`.

In [17]:
def classify(merged: list[list[RawCell]]) -> str:
    cells = [c for r in merged for c in r]
    if not cells:
        return "layout"
    if sum(c.has_link for c in cells) >= 0.5 * len(cells):
        return "toc"
    if any(c.fact_ids for c in cells):
        return "financial"
    lstart = label_start(merged)  # ignore the label column: exhibit numbers like "3.1" are not data
    values = [c for c in cells if c.start != lstart]
    numeric = sum(c.ctype == "NUMBER" for c in values)  # dashes alone (e.g. "—" trading symbol) don't make a table numeric
    return "financial" if values and numeric / len(values) >= CONFIG["numeric_table_ratio"] else "textual"


def label_start(merged) -> int:
    starts = [r[0].start for r in merged if len(r) > 1 and any(c.ctype in ("NUMBER", "DASH") or c.fact_ids for c in r[1:])]
    return Counter(starts).most_common(1)[0][0] if starts else 0


def header_row_count(merged, lstart: int) -> int:
    n = 0
    for i, cells in enumerate(merged[: CONFIG["max_header_rows"]]):
        if not cells:
            n = i + 1
            continue
        if any(c.fact_ids or c.ctype == "NUMBER" for c in cells):
            break
        units_only = len(cells) == 1 and UNITS_RE.search(cells[0].text)
        label_empty = cells[0].start > lstart
        multi = len(cells) >= 2 and (all(c.bold for c in cells) or any(c.ctype in ("YEAR", "DATE") for c in cells[1:]))
        if units_only or label_empty or multi:
            n = i + 1
            continue
        break
    return n


lstart = label_start(demo_merged)
n_header = header_row_count(demo_merged, lstart)
print("kind:", classify(demo_merged), "| label column starts at:", lstart, "| header rows:", n_header)
for r in demo_merged[:n_header]:
    print("  header:", [(c.start, c.span, c.text) for c in r])

kind: financial | label column starts at: 0 | header rows: 3
  header: []
  header: [(3, 15, 'Years ended')]
  header: [(3, 3, 'September 27, 2025'), (9, 3, 'September 28, 2024'), (15, 3, 'September 30, 2023')]


## 12. Table step 4: logical columns from content, not from widths
**What:**
1. Take the `[start, end)` interval of every *value* cell (numbers, dashes, XBRL cells) and every label cell in the data rows, and **merge overlapping intervals**. Each merged interval is one logical column. Text cells that don't overlap any of those (exhibit descriptions, "January 2028", "Not applicable") then form their own columns.
2. Project every cell onto those columns. Header cells spanning several columns (e.g. "Years ended" over 3 years) are copied to each, then stacked top-to-bottom → `"Years ended September 27, 2025"`.
3. Count **conflicts** (two values landing in one column, a data cell spanning columns, orphans in gaps).

**Why:**
- Spacer columns never contain values, so they disappear automatically, whether the layout has 18 or 57 physical columns. No widths, no hard-coded positions.
- Columns are derived from **data rows** because headers are the least regular part of a table (spanning, multi-row).
- Value cells go first, text cells second. A first version used only value cells: the text columns of the debt table (*interest rate*, *maturity date*) and of the exhibit index were then squeezed into neighbouring columns, which the conflict counter flagged.
- Flattening multi-row headers into one name per column gives each value a self-contained header ("Year Ended December 31, 2025" rather than just "2025").
- Conflicts feed the quality score, so a badly aligned table is flagged instead of silently embedded.

In [18]:
def merge_intervals(iv) -> list[list[int]]:
    groups: list[list[int]] = []
    for s, e in sorted(iv):
        if groups and s < groups[-1][1]:
            groups[-1][1] = max(groups[-1][1], e)
        else:
            groups.append([s, e])
    return groups


def column_groups(data_rows, lstart: int) -> list[list[int]]:
    multi = [c for r in data_rows if len(r) > 1 for c in r]  # single-cell rows are headings/notes
    primary = merge_intervals([(c.start, c.start + c.span) for c in multi
                               if c.ctype in ("NUMBER", "DASH") or c.fact_ids or c.start == lstart])
    # text columns (descriptions, dates, "Not applicable") that don't overlap a value column get their own column
    rest = [(c.start, c.start + c.span) for c in multi
            if not any(c.start < ge and gs < c.start + c.span for gs, ge in primary)]
    groups = sorted(primary + merge_intervals(rest))
    return groups or merge_intervals([(c.start, c.start + c.span) for r in data_rows for c in r])


def project(cells, groups, is_header: bool):
    out, conflicts = [[] for _ in groups], 0
    for c in cells:
        s, e = c.start, c.start + c.span
        hits = [j for j, (gs, ge) in enumerate(groups) if s < ge and gs < e]
        if not hits:
            hits = [min(range(len(groups)), key=lambda j: min(abs(groups[j][0] - e), abs(groups[j][1] - s)))]
            conflicts += 1
        if not is_header and len(hits) > 1:
            conflicts += 1
        for j in hits if is_header else hits[:1]:
            if out[j] and not is_header:
                conflicts += 1
            out[j].append(c)
    return out, conflicts


def column_names(header_rows, groups) -> list[str]:
    parts = [[] for _ in groups]
    for cells in header_rows:
        cells = [c for c in cells if not UNITS_ONLY_RE.match(c.text)]  # drop "(in millions)", keep "Number of RSUs (in thousands)"
        proj, _ = project(cells, groups, is_header=True)
        for j, cs in enumerate(proj):
            for c in cs:
                if not parts[j] or parts[j][-1] != c.text:
                    parts[j].append(c.text)
    return [clean(" ".join(p)) for p in parts]


demo_groups = column_groups([r for r in demo_merged[n_header:] if r], lstart)
print("logical column intervals:", demo_groups)
print("column names:", column_names(demo_merged[:n_header], demo_groups))

logical column intervals: [[0, 3], [3, 5], [9, 11], [15, 17]]
column names: ['', 'Years ended September 27, 2025', 'Years ended September 28, 2024', 'Years ended September 30, 2023']


## 13. Table step 5: row hierarchy (label → level → path)
**What:**
- A row with a label and no values is a **heading** ("Net sales:", bold "Revenues"). A long label with no values is a **note** (a footnote row), which never becomes a parent.
- Keep a stack of open headings. A heading's group **closes** when:
  1. a row is indented **no deeper** than the heading. Each heading learns its style from its first child: *nested* (children deeper, close on a row that isn't deeper) or *flat* (children at the same indent or even less; close only on a row shallower than the first child, the next heading or a total), or
  2. its matching **total** appears: `Total net sales` closes `Net sales`; `Subtotal` / `Total` close the innermost group *if it has ≥ 2 rows to add up*, otherwise a bare `Total` is a grand total. A matching total always joins its group first, even if it isn't indented deeper (Tesla's "Total other comprehensive income (loss)"). Footnote markers like `(2)(3)` are ignored when matching.

The **independent checker** (`scripts/check_coverage.py`) verifies these rules arithmetically: every total must equal the sum of its group. The per-heading mode, `Subtotal` and the grand-total rule were added because it caught real errors (Apple's fair-value table put "Level 2" items under "Level 1").
- `path` = open headings + the row's own label, e.g. `["Operating expenses", "Research and development"]`; `level` = depth in that path.

**Why:**
- "Products" on its own is ambiguous (products *net sales* or products *cost of sales*?). The path removes the ambiguity, and it's what gets embedded later.
- **I tried "cluster indents into levels" first, and it failed on Apple:** `Gross margin` is indented 37pt, *deeper* than the items above it, because Apple aligns totals visually rather than by hierarchy. Absolute indentation is not reliable, so it's used only *relative to the open heading*.
- The **total-matching rule** is accounting convention, not company-specific: a subtotal named "Total X" ends group X. It also keeps `Total automotive revenues` *inside* `Revenues` (it doesn't match "Revenues"), which is correct for Tesla.
- Only headings become parents. Making every row a parent would nest items under totals.
- Fallback for tables with no indentation at all: headings stay open until the next heading or matching total.

In [19]:
def to_cell(cs: list[RawCell]) -> Cell:
    if not cs:
        return Cell("", "EMPTY")
    text = clean(" ".join(c.text for c in cs))
    ctype = cell_type(text) if len(cs) > 1 else cs[0].ctype
    v, pct, cur, _ = parse_number(text) if ctype in ("NUMBER", "DASH") else (None, False, None, False)
    if ctype == "YEAR":
        v = float(text[-4:])
    return Cell(text, ctype, v, pct, cur, [f for c in cs for f in c.fact_ids])


TOTAL_RE = re.compile(r"^(sub)?total\b", re.I)
FOOTNOTE_RE = re.compile(r"\((?:\d{1,2}|[a-z])\)|\*+$", re.I)


def norm_label(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", FOOTNOTE_RE.sub("", s).lower()).strip()


def closes_group(total_label: str, heading: str) -> bool:
    rest = re.sub(r"^(sub)?total\s*", "", norm_label(total_label))
    return rest == "" or rest == norm_label(heading)


@dataclass
class OpenGroup:
    indent: float
    label: str
    mode: str | None = None  # "nested" (children indented deeper) or "flat" (children at the same indent or less)
    child_indent: float = 0.0  # indent of the first child; a flat group closes on a row shallower than this
    n_data: int = 0


def build_rows(data_rows, groups):
    projected, conflicts = [], 0
    for cells in data_rows:
        if cells:
            proj, k = project(cells, groups, is_header=False)
            projected.append(proj)
            conflicts += k if len(cells) > 1 else 0
    tol = CONFIG["indent_tolerance_pt"]
    rows, stack = [], []  # open heading groups
    for proj in projected:
        label_cells = proj[0]
        label = clean(" ".join(c.text for c in label_cells)).rstrip(":").strip()
        values = [to_cell(cs) for cs in proj[1:]]
        has_values = any(v.ctype != "EMPTY" for v in values)
        if not has_values and len(label) >= CONFIG["note_min_len"]:
            rows.append(Row(label, 0, [label], "note", False, values))
            continue
        if not label and any(v.ctype in ("NUMBER", "DASH") for v in values) and rows and rows[-1].row_type == "data":
            label = rows[-1].label  # continuation row: Apple prints "Operating leases" once for 2 lines
        row_type = "heading" if label and not has_values else "data"
        indent = label_cells[0].indent_pt if label_cells else math.inf  # unlabeled row continues the group
        is_total = bool(TOTAL_RE.match(label))
        bare = is_total and closes_group(label, "")  # "Total", "Subtotal", "Total (2)(3)"
        owner = None
        if is_total:  # a bare total only closes a group that has something to add up
            owner = next((i for i in range(len(stack) - 1, -1, -1)
                          if closes_group(label, stack[i].label) and (not bare or stack[i].n_data >= 2)), None)
        if owner is not None:  # "Total X" belongs to open group X even when it isn't indented deeper
            del stack[owner + 1:]
        elif bare:  # bare total with no group to close: grand total
            stack.clear()
        elif row_type == "heading":
            while stack and indent - stack[-1].indent <= tol:
                stack.pop()
        else:
            if stack and stack[-1].mode is None:  # the first child always belongs to its heading and sets the style
                stack[-1].mode = "nested" if indent - stack[-1].indent > tol else "flat"
                stack[-1].child_indent = indent
            while stack and ((stack[-1].mode == "nested" and indent - stack[-1].indent <= tol)
                             or (stack[-1].mode == "flat" and indent < stack[-1].child_indent - tol)):
                stack.pop()
        path = [s.label for s in stack] + ([label] if label else [])
        rows.append(Row(label, len(stack), path, row_type, is_total, values))
        if row_type == "heading":
            stack.append(OpenGroup(indent, label))
        else:
            if stack:
                stack[-1].n_data += 1
            if owner is not None:
                stack.pop()
    return rows, conflicts


demo_rows, demo_conflicts = build_rows(demo_merged[n_header:], demo_groups)
print("conflicts:", demo_conflicts)
pd.DataFrame([{"type": r.row_type, "level": r.level, "path": " > ".join(r.path), "values": [c.text for c in r.cells]}
              for r in demo_rows]).head(14)

conflicts: 0


,type,level,path,values
0,heading,0,Net sales,"[, , ]"
1,data,1,Net sales > Products,"[$307,003, $294,866, $298,085]"
2,data,1,Net sales > Services,"[109,158, 96,169, 85,200]"
3,data,1,Net sales > Total net sales,"[416,161, 391,035, 383,285]"
4,heading,0,Cost of sales,"[, , ]"
5,data,1,Cost of sales > Products,"[194,116, 185,233, 189,282]"
6,data,1,Cost of sales > Services,"[26,844, 25,119, 24,855]"
7,data,1,Cost of sales > Total cost of sales,"[220,960, 210,352, 214,137]"
8,data,0,Gross margin,"[195,201, 180,683, 169,148]"
9,heading,0,Operating expenses,"[, , ]"


## 14. Table step 6: title, units and intro sentence
**What:** look back at up to 4 text blocks *since the previous table*:
- **units** = a units line like "(In millions, except per-share amounts)" kept verbatim, or just the canonical phrase ("dollars in millions") when it sits inside a sentence; fallback = the majority XBRL `scale` of the table's facts;
- **title** = the nearest short **bold** line, skipping the units line and the company name;
- **intro** = the nearest non-bold sentence just before the table ("The following table shows net sales by reportable segment… (dollars in millions):").

**Why:**
- The title and units live **outside** the `<table>`. Without them, `307,003` is meaningless to an LLM: is it dollars? millions? which statement?
- "Since the previous table" stops a title from leaking from one table to the next.
- The XBRL `scale` fallback covers MD&A tables where the units are stated only in a sentence far away.
- The intro sentence often explains the table better than the title does, so it gets stored for chunk context later.

In [20]:
def is_units_line(t: str) -> bool:
    return bool(UNITS_RE.search(t)) and t.startswith("(") and t.endswith(")")


def table_context(prev: list[tuple[str, bool]], header_text: str, fact_list: list[XbrlFact], company: str):
    cands = prev[-CONFIG["title_lookback"]:][::-1]  # nearest first
    units = next((t if is_units_line(t) else m.group(0) for t, _ in cands if (m := UNITS_RE.search(t))), None)
    if not units and (m := UNITS_RE.search(header_text)):
        units = next((p for p in re.findall(r"\([^()]*\)", header_text) if UNITS_RE.search(p)), m.group(0))
    if not units:
        scales = Counter(f.scale for f in fact_list if f.unit and f.unit.lower() == "usd")
        units = SCALE_TO_UNITS.get(scales.most_common(1)[0][0]) if scales else None
    usable = [(t, b) for t, b in cands if not is_units_line(t) and norm_label(t) != norm_label(company)]
    title = next((t for t, b in usable if b and len(t) <= CONFIG["heading_max_len"]), None)
    before_title = usable[: next((i for i, (t, _) in enumerate(usable) if t == title), len(usable))]
    intro = next((t for t, b in before_title[:2] if not b and len(t) > 40), None)  # only text between title and table
    return title, units, intro

## 15. Table step 7: validate the parse against XBRL, decide period axis, score quality
**What:** for every cell carrying an XBRL fact, check:
- **value_match**: the parsed number equals the XBRL number (catches bad merges or misplaced cells);
- **sign_match**: parentheses/minus agree with `sign="-"` (informational: filers sometimes print expenses as positive);
- **period purity** per column *and* per row: do all facts in a column share one period? Whichever axis is purer is the **period axis** (normally columns; sometimes rows; in "as of one date" tables there is none);
- **concept purity**: do all facts in a row (or in a column) describe one concept? The better orientation is used, because some tables have one concept per row (income statement) and others one concept per column (a debt table: *principal | rate | maturity*);
- **header_period_agreement** (only when periods really vary across columns): does the column header mention the year of its XBRL period?

Then combine these and the structural signals (header found, title, units, conflicts) into a **quality score in 0–1**, with reasons.

**Why:**
- This is an **automatic test that needs no hand labels**. On roughly 75% of numeric tables, XBRL tells us exactly what each cell should be, so a parser bug shows up as a number, not as a wrong answer three weeks later.
- Detecting the period axis handles the "different structure" problem: the same code copes with years-as-columns and years-as-rows.
- XBRL periods also **fill missing column names** and attach `period_start`/`period_end` to each column, which is useful metadata for chunks.
- Low-scoring tables are flagged `low_confidence`. They're kept, but listed in a report for review. Never silent.

In [21]:
def purity(groups: dict) -> float | None:
    total = sum(sum(c.values()) for c in groups.values())
    return None if not total else sum(c.most_common(1)[0][1] for c in groups.values()) / total


def validate(rows: list[Row], columns: list[Column], facts: dict, contexts: dict) -> dict:
    by_col_p, by_row_p, by_col_c, by_row_c = (defaultdict(Counter) for _ in range(4))
    n = n_value = value_ok = sign_ok = 0
    for i, r in enumerate(rows):
        for j, c in enumerate(r.cells, start=1):
            f = facts.get(c.fact_ids[0]) if c.fact_ids else None
            if f is None:
                continue
            n += 1
            # only numeric cells: ranges ("4.70-5.75%", 2 facts) and prose ("20 million vehicles delivered") are kept verbatim
            if len(c.fact_ids) == 1 and f.shown is not None and c.ctype in ("NUMBER", "DASH"):
                n_value += 1
                if c.value is not None and any(math.isclose(abs(c.value), abs(x), rel_tol=1e-9, abs_tol=1e-9)
                                               for x in (f.shown, f.value)):
                    value_ok += 1
            if c.value is not None and (c.value == 0 or (c.value < 0) == (f.sign < 0)):
                sign_ok += 1
            ctx = contexts.get(f.context, {})
            p = (ctx.get("start"), ctx.get("end"))
            by_col_p[j][p] += 1; by_row_p[i][p] += 1
            by_col_c[j][f.concept] += 1; by_row_c[i][f.concept] += 1
    if not n:
        return {"n_fact_cells": 0}
    col_p, row_p = purity(by_col_p), purity(by_row_p)
    n_periods = len({p for cnt in by_col_p.values() for p in cnt})
    axis = None if n_periods == 1 else ("columns" if col_p >= row_p else "rows")
    agree = checked = 0
    for j, cnt in by_col_p.items():
        (start, end), _ = cnt.most_common(1)[0]
        if axis == "columns" or n_periods == 1:
            columns[j].period_start, columns[j].period_end = start, end
        if axis == "columns" and end and columns[j].name:
            checked += 1
            agree += end[:4] in columns[j].name
        elif axis == "columns" and end:
            columns[j].name = f"{start} to {end}" if start else f"As of {end}"
    return {
        "n_fact_cells": n,
        "value_match": round(value_ok / n_value, 3) if n_value else None,
        "sign_match": round(sign_ok / n, 3),
        "period_axis": axis,
        "period_purity": round(max(col_p, row_p), 3),
        "concept_purity": round(max(purity(by_row_c), purity(by_col_c)), 3),
        "header_period_agreement": round(agree / checked, 3) if checked else None,
    }


def score(kind, n_header, title, units, conflicts, n_cells, v) -> dict:
    s, reasons = 1.0, []
    if kind == "financial" and n_header == 0:
        s -= 0.3; reasons.append("no_header")
    if kind == "financial" and not units:
        s -= 0.1; reasons.append("no_units")
    if not title:
        s -= 0.1; reasons.append("no_title")
    if conflicts:
        s -= min(0.4, 2 * conflicts / max(1, n_cells)); reasons.append(f"conflicts={conflicts}")
    if v.get("n_fact_cells"):
        for key, w in (("value_match", 0.4), ("period_purity", 0.3), ("concept_purity", 0.2), ("header_period_agreement", 0.2)):
            if v.get(key) is not None and v[key] < 1:
                s -= w * (1 - v[key]); reasons.append(f"{key}={v[key]}")
    s = round(max(0.0, s), 3)
    return {**v, "score": s, "low_confidence": s < CONFIG["low_confidence"], "reasons": reasons}

## 16. Render a table to Markdown (one *view* of the structured table)
**What:** a Markdown table whose first column is the full row **path**, with numbers normalized (`(1,234)` → `-1,234`, `$307,003` → `307,003`) and a bold title + units line on top. Heading rows are folded into paths; note rows go below the table.

**Why:**
- Markdown is what we will hand to the LLM. LLMs read Markdown tables well, and they're compact.
- Using the **path** as the label means every row is self-explanatory even if a chunk later contains only part of the table.
- Normalizing numbers makes values consistent across companies (no `$` and `(` fragments). The exact original text is still stored in `Cell.text`.
- This is only a *view*. The JSON structure stays the source of truth, and other views (row sentences for embedding) can be generated from it during chunking.

In [22]:
def fmt_value(c: Cell) -> str:
    if c.ctype == "DASH":
        return "—"
    if c.value is None or c.ctype == "YEAR":
        return c.text.replace("|", "\\|")
    m = re.search(r"\.(\d+)", c.text)
    return f"{c.value:,.{len(m.group(1)) if m else 0}f}" + ("%" if c.is_percent else "")


def to_markdown(t: Table) -> str:
    units = t.units if not t.units or t.units.startswith("(") else f"({t.units})"
    head = f"**{t.title or 'Table'}**" + (f" {units}" if units and t.units not in (t.title or "") else "")
    if t.kind == "text_block":
        return "\n".join(r.label for r in t.rows)
    names = [t.columns[0].name or "Line item"] + [c.name or f"Column {j}" for j, c in enumerate(t.columns[1:], 1)]
    lines = [head, "", "| " + " | ".join(n.replace("|", "\\|") for n in names) + " |", "|" + "---|" * len(names)]
    notes = []
    for r in t.rows:
        if r.row_type == "note":
            notes.append(r.label)
        elif r.row_type == "data":
            label = " > ".join(r.path).replace("|", "\\|")
            lines.append("| " + " | ".join([label] + [fmt_value(c) for c in r.cells]) + " |")
    return "\n".join(lines + ([""] + [f"_Note: {n}_" for n in notes] if notes else []))

## 17. Assemble: `parse_table()` and `parse_document()`
**What:** chain steps 9–16 per table, and steps 5–8 per document, producing a `Document` with ordered `Element`s.

**Why:**
- The orchestration is a short, flat sequence of small functions. Each step can be unit-tested on its own, and when one company breaks you can see exactly which step failed.
- `layout` and `toc` tables are **dropped but counted**. Single-column "tables" (Workiva uses them for footnotes and bullets) become **paragraphs**, because that's what they are semantically.
- Every element keeps `doc_id`, `part`, `section` and `section_title`, so chunking later needs no access to the original HTML.

In [23]:
def parse_table(tag, table_id, prev_text, meta, facts, contexts) -> Table:
    merged = [merge_fragments(r) for r in raw_rows(tag)]
    kind = classify(merged)
    if kind in ("layout", "toc"):
        return Table(table_id, kind)
    lstart = label_start(merged)
    n_header = header_row_count(merged, lstart)
    if not any(merged[n_header:]):  # every row looked like a header: it's a small text grid, keep all rows as data
        n_header = 0
    header_rows, data_rows = merged[:n_header], merged[n_header:]
    groups = column_groups([r for r in data_rows if r], lstart)
    if not groups:
        return Table(table_id, "layout")
    rows, conflicts = build_rows(data_rows, groups)
    if len(groups) == 1:
        return Table(table_id, "text_block", rows=rows, markdown="\n".join(r.label for r in rows))
    columns = [Column(n) for n in column_names(header_rows, groups)]
    v = validate(rows, columns, facts, contexts)
    fact_list = [facts[f] for r in rows for c in r.cells for f in c.fact_ids if f in facts]
    header_text = " ".join(c.text for r in header_rows for c in r)
    title, units, intro = table_context(prev_text, header_text, fact_list, meta["company"])
    n_cells = sum(len(r) for r in data_rows)
    t = Table(table_id, kind, title, units, intro, columns, rows, v.get("period_axis"),
              score(kind, n_header, title, units, conflicts, n_cells, v))
    t.markdown = to_markdown(t)
    return t


def parse_document(path: Path):
    soup = load(path)
    meta = doc_metadata(soup, path)
    contexts = parse_contexts(soup)
    facts = {f.id: f for f in map(parse_fact, soup.find_all("ix:nonfraction"))}
    removed = strip_noise(soup)
    blocks = list(iter_blocks(soup.body))
    boilerplate = find_boilerplate([clean(b.get_text("")) for b in blocks if b.name != "table"])

    doc = Document(**meta)
    part, section, section_title = "Front Matter", "Cover Page", "Cover Page"
    pending_title, prev_text, n_tables = False, [], 0

    def add(etype, text=None, table=None):
        doc.elements.append(Element(meta["doc_id"], len(doc.elements), etype, part, section, section_title, text, table))

    for b in blocks:
        if b.name == "table":
            n_tables += 1
            t = parse_table(b, f"{meta['doc_id']}_t{n_tables:03d}", prev_text, meta, facts, contexts)
            prev_text = []
            if t.kind in ("layout", "toc"):
                removed[f"{t.kind}_tables"] += 1
            elif t.kind == "text_block":
                add("paragraph", text=t.markdown)
            else:
                add("table", table=t)
            continue
        text = clean(b.get_text(""))
        bold = is_bold(b)
        if is_boilerplate(text, boilerplate, bold):
            removed["boilerplate_lines"] += 1
            continue
        sec = detect_section(text, bold)
        if sec and sec[0] == "part":
            part, section, section_title, etype = sec[1], None, sec[1], "heading"
        elif sec:
            section, section_title, pending_title = sec[1], sec[2], sec[2] is None
            etype = "heading"
        elif pending_title and bold and len(text) <= CONFIG["heading_max_len"]:
            section_title, pending_title, etype = text, False, "heading"
        else:
            etype = "heading" if bold and len(text) <= CONFIG["heading_max_len"] else "paragraph"
        add(etype, text=text)
        prev_text.append((text, bold))
    return doc, facts, contexts, removed

## 18. Run on every filing and save outputs
**What:** parse all files, then write per document:
- `<doc_id>.document.json`: document metadata
- `<doc_id>.elements.jsonl`: one element per line (heading / paragraph / table with full structure + Markdown)
- `<doc_id>.facts.jsonl`: every XBRL numeric fact with period and dimensions
- `quality_report.csv`: one line per table with score and reasons

**Why:**
- **JSONL** streams line by line, diffs cleanly in git, and is what most chunking/embedding tools accept. One element per line means a downstream bug affects one line, not the whole file.
- Facts are kept separate from text: they're for exact lookups (and future text-to-SQL), not for embedding.
- The quality report is the **review queue**. When you add a new company, read this first.

In [24]:
def save(doc: Document, facts: dict, contexts: dict) -> None:
    meta = {k: v for k, v in asdict(doc).items() if k != "elements"}
    (OUT_DIR / f"{doc.doc_id}.document.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    with open(OUT_DIR / f"{doc.doc_id}.elements.jsonl", "w", encoding="utf-8") as fh:
        for e in doc.elements:
            fh.write(json.dumps(asdict(e), ensure_ascii=False) + "\n")
    facts_frame(facts, contexts).assign(doc_id=doc.doc_id).to_json(
        OUT_DIR / f"{doc.doc_id}.facts.jsonl", orient="records", lines=True, force_ascii=False)


docs, summary, quality_rows = {}, [], []
for path in FILES:
    doc, facts, contexts, removed = parse_document(path)
    save(doc, facts, contexts)
    docs[doc.doc_id] = doc
    types = Counter(e.type for e in doc.elements)
    tables = [e.table for e in doc.elements if e.table]
    summary.append({"doc_id": doc.doc_id, "company": doc.company, "fy": doc.fiscal_year, **types,
                    "xbrl_facts": len(facts), **removed,
                    "low_conf_tables": sum(t.quality.get("low_confidence", False) for t in tables)})
    for e in doc.elements:
        if e.table:
            t = e.table
            quality_rows.append({"doc_id": doc.doc_id, "table_id": t.table_id, "section": e.section, "kind": t.kind,
                                 "title": t.title, "units": t.units, "rows": len(t.rows), "cols": len(t.columns),
                                 **{k: t.quality.get(k) for k in ("score", "n_fact_cells", "value_match", "period_purity",
                                                                    "concept_purity", "header_period_agreement")},
                                 "reasons": ";".join(t.quality.get("reasons", []))})

quality = pd.DataFrame(quality_rows)
quality.to_csv(OUT_DIR / "quality_report.csv", index=False)
pd.DataFrame(summary).fillna(0)

,doc_id,company,fy,heading,paragraph,table,xbrl_facts,hidden_blocks,page_header_footer_blocks,layout_tables,boilerplate_lines,toc_tables,low_conf_tables
0,AAPL_10-K_2025,Apple Inc.,2025,148,425,51,969,33,60.0,8.0,13,3.0,0
1,MSFT_10-K_2026,MICROSOFT CORPORATION,2026,180,702,85,1583,1,0.0,0.0,299,0.0,0
2,TSLA_10-K_2025,"Tesla, Inc.",2025,193,690,74,1477,317,105.0,2.0,55,2.0,0


## 19. Inspect results
### 19a. Sections detected per document
**Why:** if a filer's headings aren't detected, every element in it gets the wrong `section`. This table is the quickest sanity check.

In [25]:
pd.DataFrame([{"doc_id": d.doc_id, "part": e.part, "section": e.section, "title": e.section_title,
               "elements": n} for d in docs.values()
              for (e, n) in [(g[0], len(g)) for g in
                             [[x for x in d.elements if (x.part, x.section) == key]
                              for key in dict.fromkeys((x.part, x.section) for x in d.elements)]]])

,doc_id,part,section,title,elements
0,AAPL_10-K_2025,Front Matter,Cover Page,Cover Page,43
1,AAPL_10-K_2025,Part I,NaN,Part I,1
2,AAPL_10-K_2025,Part I,Item 1,Business,53
3,AAPL_10-K_2025,Part I,Item 1A,Risk Factors,107
4,AAPL_10-K_2025,Part I,Item 1B,Unresolved Staff Comments,2
...,...,...,...,...,...
79,TSLA_10-K_2025,Part III,Item 13,CERTAIN RELATIONSHIPS AND RELATED TRANSACTIONS AND DIRECTOR INDEPENDENCE,2
80,TSLA_10-K_2025,Part III,Item 14,PRINCIPAL ACCOUNTANT FEES AND SERVICES,2
81,TSLA_10-K_2025,Part IV,NaN,Part IV,1
82,TSLA_10-K_2025,Part IV,Item 15,EXHIBITS AND FINANCIAL STATEMENT SCHEDULES,15


### 19b. Table quality overview
**Why:** score distribution per kind, plus the tables that need review. `value_match = 1.0` and high purity mean that table's grid matches XBRL exactly.

In [26]:
display(quality.groupby(["doc_id", "kind"]).agg(tables=("table_id", "count"), mean_score=("score", "mean"),
                                                  min_score=("score", "min"), xbrl_cells=("n_fact_cells", "sum")).round(3))
quality.sort_values("score").head(10)[["table_id", "section", "kind", "title", "rows", "cols", "score", "reasons"]]

tables  mean_score  min_score  xbrl_cells
doc_id         kind                                                
AAPL_10-K_2025 financial      40       0.944      0.767         868
               textual        11       0.906      0.667           0
MSFT_10-K_2026 financial      69       0.910      0.738        1420
               textual        16       0.944      0.900           0
TSLA_10-K_2025 financial      59       0.919      0.700        1223
               textual        15       0.947      0.900           0

,table_id,section,kind,title,rows,cols,score,reasons
49,AAPL_10-K_2025_t061,Item 16,textual,SIGNATURES,4,2,0.667,conflicts=1
186,TSLA_10-K_2025_t056,Item 8,financial,2025 CEO Performance Award,13,5,0.700,no_units;header_period_agreement=0.0
100,MSFT_10-K_2026_t052,Item 8,financial,NaN,20,12,0.738,no_title;concept_purity=0.857;header_period_agreement=0.333
167,TSLA_10-K_2025_t037,Item 8,financial,NaN,3,3,0.750,no_title;period_purity=0.5
185,TSLA_10-K_2025_t055,Item 8,financial,NaN,7,7,0.752,no_title;period_purity=0.966;concept_purity=0.31
97,MSFT_10-K_2026_t049,Item 8,financial,NaN,5,7,0.767,no_title;header_period_agreement=0.333
30,AAPL_10-K_2025_t041,Item 8,financial,NaN,9,4,0.767,no_title;concept_purity=0.333
158,TSLA_10-K_2025_t028,Item 8,financial,NaN,5,3,0.780,no_title;period_purity=0.6
164,TSLA_10-K_2025_t034,Item 8,financial,NaN,5,4,0.780,no_title;period_purity=0.6
118,MSFT_10-K_2026_t070,Item 8,financial,NaN,21,4,0.783,no_title;period_purity=0.611


### 19c. Final output: Apple, Consolidated Statements of Operations
Compare with the raw HTML you started from: 18 physical columns, split `$` cells, a 2-row header and `&nbsp;` indentation are now **4 clean columns, flattened headers and full row paths**.

In [27]:
def find_table(doc_id: str, pattern: str) -> Table:
    return next(e.table for e in docs[doc_id].elements if e.table and re.search(pattern, e.table.title or "", re.I))


apple_id = next(d for d in docs if d.startswith("AAPL"))
tesla_id = next(d for d in docs if d.startswith("TSLA"))
display(Markdown(find_table(apple_id, r"statements of operations").markdown))

**CONSOLIDATED STATEMENTS OF OPERATIONS** (In millions, except number of shares, which are reflected in thousands, and per-share amounts)

| Line item | Years ended September 27, 2025 | Years ended September 28, 2024 | Years ended September 30, 2023 |
|---|---|---|---|
| Net sales > Products | 307,003 | 294,866 | 298,085 |
| Net sales > Services | 109,158 | 96,169 | 85,200 |
| Net sales > Total net sales | 416,161 | 391,035 | 383,285 |
| Cost of sales > Products | 194,116 | 185,233 | 189,282 |
| Cost of sales > Services | 26,844 | 25,119 | 24,855 |
| Cost of sales > Total cost of sales | 220,960 | 210,352 | 214,137 |
| Gross margin | 195,201 | 180,683 | 169,148 |
| Operating expenses > Research and development | 34,550 | 31,370 | 29,915 |
| Operating expenses > Selling, general and administrative | 27,601 | 26,097 | 24,932 |
| Operating expenses > Total operating expenses | 62,151 | 57,467 | 54,847 |
| Operating income | 133,050 | 123,216 | 114,301 |
| Other income/(expense), net | -321 | 269 | -565 |
| Income before provision for income taxes | 132,729 | 123,485 | 113,736 |
| Provision for income taxes | 20,719 | 29,749 | 16,741 |
| Net income | 112,010 | 93,736 | 96,995 |
| Earnings per share > Basic | 7.49 | 6.11 | 6.16 |
| Earnings per share > Diluted | 7.46 | 6.08 | 6.13 |
| Shares used in computing earnings per share > Basic | 14,948,500 | 15,343,783 | 15,744,231 |
| Shares used in computing earnings per share > Diluted | 15,004,697 | 15,408,095 | 15,812,547 |

### 19d. Final output: Tesla, Consolidated Statements of Operations
**Same code, different structure:** Tesla uses `padding-left` instead of `&nbsp;`, bold section rows instead of "Net sales:", calendar years and a different header layout.

In [28]:
display(Markdown(find_table(tesla_id, r"statements of operations").markdown))

**Consolidated Statements of Operations** (in millions, except per share data)

| Line item | Year Ended December 31, 2025 | Year Ended December 31, 2024 | Year Ended December 31, 2023 |
|---|---|---|---|
| Revenues > Automotive sales | 65,821 | 72,480 | 78,509 |
| Revenues > Automotive regulatory credits | 1,993 | 2,763 | 1,790 |
| Revenues > Automotive leasing | 1,712 | 1,827 | 2,120 |
| Revenues > Total automotive revenues | 69,526 | 77,070 | 82,419 |
| Revenues > Energy generation and storage | 12,771 | 10,086 | 6,035 |
| Revenues > Services and other | 12,530 | 10,534 | 8,319 |
| Revenues > Total revenues | 94,827 | 97,690 | 96,773 |
| Cost of revenues > Automotive sales | 56,267 | 61,870 | 65,121 |
| Cost of revenues > Automotive leasing | 898 | 1,003 | 1,268 |
| Cost of revenues > Total automotive cost of revenues | 57,165 | 62,873 | 66,389 |
| Cost of revenues > Energy generation and storage | 8,969 | 7,446 | 4,894 |
| Cost of revenues > Services and other | 11,599 | 9,921 | 7,830 |
| Cost of revenues > Total cost of revenues | 77,733 | 80,240 | 79,113 |
| Gross profit | 17,094 | 17,450 | 17,660 |
| Operating expenses > Research and development | 6,411 | 4,540 | 3,969 |
| Operating expenses > Selling, general and administrative | 5,834 | 5,150 | 4,800 |
| Operating expenses > Restructuring and other | 494 | 684 | — |
| Operating expenses > Total operating expenses | 12,739 | 10,374 | 8,769 |
| Income from operations | 4,355 | 7,076 | 8,891 |
| Interest income | 1,680 | 1,569 | 1,066 |
| Interest expense | -338 | -350 | -156 |
| Other (expense) income, net | -419 | 695 | 172 |
| Income before income taxes | 5,278 | 8,990 | 9,973 |
| Provision for (benefit from) income taxes | 1,423 | 1,837 | -5,001 |
| Net income | 3,855 | 7,153 | 14,974 |
| Net income (loss) attributable to noncontrolling interests and redeemable noncontrolling interests in subsidiaries | 61 | 62 | -23 |
| Net income attributable to common stockholders | 3,794 | 7,091 | 14,997 |
| Net income per share of common stock attributable to common stockholders > Basic | 1.18 | 2.23 | 4.73 |
| Net income per share of common stock attributable to common stockholders > Diluted | 1.08 | 2.04 | 4.30 |
| Weighted average shares used in computing net income per share of common stock > Basic | 3,225 | 3,197 | 3,174 |
| Weighted average shares used in computing net income per share of common stock > Diluted | 3,528 | 3,498 | 3,485 |

### 19e. Final output: an MD&A table (no title tag, units in a sentence) and its validation report
**Why:** MD&A tables are the hardest case: their titles are bold paragraph headings and their units sit in the intro sentence. The `intro` field captures that sentence for chunk context.

In [29]:
mdna = next(e.table for e in docs[apple_id].elements if e.table and e.section == "Item 7" and e.table.kind == "financial")
print("title:", mdna.title, "| units:", mdna.units, "\nintro:", mdna.intro, "\nquality:", mdna.quality)
display(Markdown(mdna.markdown))

title: Segment Operating Performance | units: dollars in millions 
intro: The following table shows net sales by reportable segment for 2025, 2024 and 2023 (dollars in millions): 
quality: {'n_fact_cells': 0, 'score': 1.0, 'low_confidence': False, 'reasons': []}


**Segment Operating Performance** (dollars in millions)

| Line item | 2025 | Change | 2024 | Change | 2023 |
|---|---|---|---|---|---|
| Americas | 178,353 | 7% | 167,045 | 3% | 162,560 |
| Europe | 111,032 | 10% | 101,328 | 7% | 94,294 |
| Greater China | 64,377 | -4% | 66,952 | -8% | 72,559 |
| Japan | 28,703 | 15% | 25,052 | 3% | 24,257 |
| Rest of Asia Pacific | 33,696 | 10% | 30,658 | 4% | 29,615 |
| Total net sales | 416,161 | 6% | 391,035 | 2% | 383,285 |

### 19f. Final output: a completely different structure (Tesla debt table)
**Why:** this table has *concepts as columns* (carrying value, principal, rate, maturity), a two-level header ("Net Carrying Value" over "Current | Long-Term"), text columns, ranges like `4.70-5.75%`, and one reporting date (so `period_axis = None`). No code was written for it specifically.

In [30]:
debt = next(e.table for e in docs[tesla_id].elements
            if e.table and any("Interest Rate" in c.name for c in e.table.columns))
print("quality:", debt.quality)
display(Markdown(debt.markdown))

quality: {'n_fact_cells': 45, 'value_match': 1.0, 'sign_match': 0.889, 'period_axis': None, 'period_purity': 1.0, 'concept_purity': 0.911, 'header_period_agreement': None, 'score': 0.982, 'low_confidence': False, 'reasons': ['concept_purity=0.911']}


**Note 9 – Debt** (in millions)

| Line item | Net Carrying Value Current | Net Carrying Value Long-Term | Unpaid Principal Balance | Unused Committed Amount (1) | Contractual Interest Rates | Contractual Maturity Date |
|---|---|---|---|---|---|---|
| Recourse debt > RCF Credit Agreement | — | — | — | 5,000 | Not applicable | January 2028 |
| Recourse debt > Other | 1 | 2 | 3 | — | 4.70-5.75% | January 2026- January 2031 |
| Recourse debt > Total recourse debt | 1 | 2 | 3 | 5,000 |  |  |
| Non-recourse debt > Automotive Asset-backed Notes | 1,492 | 1,745 | 3,249 | — | 2.47-6.57% | October 2026- June 2035 |
| Non-recourse debt > China Working Capital Facility | — | 4,288 | 4,288 | 1,429 | 2.01-2.11% | March 2026- December 2026 (2) |
| Non-recourse debt > Energy Asset-backed Notes | 55 | 337 | 397 | — | 5.08-6.25% | June 2050 |
| Non-recourse debt > Cash Equity Debt | 21 | 212 | 240 | — | 5.25-5.81% | July 2034- January 2035 |
| Non-recourse debt > Total non-recourse debt | 1,568 | 6,582 | 8,174 | 1,429 |  |  |
| Total debt | 1,569 | 6,584 | 8,177 | 6,429 |  |  |
| Finance leases | 71 | 152 |  |  |  |  |
| Total debt and finance leases | 1,640 | 6,736 |  |  |  |  |

### 19g. What one table element looks like in `elements.jsonl`
**Why:** this structure is the input to the chunking stage. Note the `path`, typed `value`, `fact_ids`, column periods and quality.

In [31]:
el = next(e for e in docs[apple_id].elements if e.table and re.search("statements of operations", e.table.title or "", re.I))
d = asdict(el)
d["table"]["rows"] = d["table"]["rows"][:2]
d["table"]["markdown"] = d["table"]["markdown"][:200] + " ..."
print(json.dumps(d, indent=2)[:3500])

{
  "doc_id": "AAPL_10-K_2025",
  "order": 357,
  "type": "table",
  "part": "Part II",
  "section": "Item 8",
  "section_title": "Financial Statements and Supplementary Data",
  "text": null,
  "table": {
    "table_id": "AAPL_10-K_2025_t023",
    "kind": "financial",
    "title": "CONSOLIDATED STATEMENTS OF OPERATIONS",
    "units": "(In millions, except number of shares, which are reflected in thousands, and per-share amounts)",
    "intro": null,
    "columns": [
      {
        "name": "",
        "period_start": null,
        "period_end": null
      },
      {
        "name": "Years ended September 27, 2025",
        "period_start": "2024-09-29",
        "period_end": "2025-09-27"
      },
      {
        "name": "Years ended September 28, 2024",
        "period_start": "2023-10-01",
        "period_end": "2024-09-28"
      },
      {
        "name": "Years ended September 30, 2023",
        "period_start": "2022-09-25",
        "period_end": "2023-09-30"
      }
    ],
    "row

### 19h. Output files

In [32]:
pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)} for p in sorted(OUT_DIR.iterdir())])

,file,KB
0,AAPL_10-K_2025.document.json,0.2
1,AAPL_10-K_2025.elements.jsonl,620.2
2,AAPL_10-K_2025.facts.jsonl,297.9
3,MSFT_10-K_2026.document.json,0.2
4,MSFT_10-K_2026.elements.jsonl,1032.6
5,MSFT_10-K_2026.facts.jsonl,606.6
6,quality_report.csv,23.6
7,TSLA_10-K_2025.document.json,0.2
8,TSLA_10-K_2025.elements.jsonl,990.3
9,TSLA_10-K_2025.facts.jsonl,449.8
